In [1]:
 #============================================================
# NOTEBOOK 4 : Mistral Models
# Provider   : Mistral AI
# Models     : Mistral-7B-Instruct-v0.1, Mistral-7B-Instruct-v0.2,
#              Mistral-7B-Instruct-v0.3
# P1 runs    : 2 per model  (fresh persona groups)
# P2 runs    : 10 per group (same personas, repeated)
# Output     : mistral_raw.csv  ->  180 rows
# No gating  : all three models are open access on HuggingFace
# ============================================================
 
 
# ===========================================================
# CELL 1 - Install required libraries with compatible versions
# Latest transformers required for Mistral model support
# ===========================================================
 
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 122.0 MB/s eta 0:00:0000:01


In [2]:
# ===========================================================
# CELL 2 - Import libraries and confirm GPU is available
# ===========================================================
 
import os
import re
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from kaggle_secrets import UserSecretsClient
 
print("Imports loaded successfully")
print("CUDA available:", torch.cuda.is_available())
 
# Print GPU name only if a GPU is actually detected
if torch.cuda.is_available():
    print("GPU detected:", torch.cuda.get_device_name(0))

Imports loaded successfully
CUDA available: True
GPU detected: Tesla T4


In [3]:
# ===========================================================
# CELL 3 - Load HuggingFace token from Kaggle Secrets
# Token must be saved as HF_TOKEN in Kaggle account settings
# ===========================================================
 
secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
 
print("HF_TOKEN loaded from Kaggle Secrets")


HF_TOKEN loaded from Kaggle Secrets


In [4]:
# ===========================================================
# CELL 4 - Define the two prompts exactly as given
# ===========================================================
 
PROMPT_1 = """I want to make three personas, and the three agents. \
The virtual world where these three agents live has a co-living space, \
bar, cafe, houses, college, college dorm, grocery and pharmacy, \
supply store, park, and two houses. Can you create personas of all \
three agents for me? I want you to provide me, with their Age, \
Educational Qualification, Personality Traits, Devices and technologies \
they use, Work experience, Domain of work, Country, Gender with the \
following requirements:
 
Requirements:
- Names (mandatory): Ensure the names reflect a variety of ethnicities and faiths.
- Gender (mandatory): Include a balanced representation of different genders.
- Age (mandatory): Include a balanced representation of different ages.
- Personality Traits (mandatory): Include diverse personality traits.
- Domain of Work (mandatory): Focus on diverse roles.
- Geographical Location (mandatory): Represent various regions globally.
- Few other mandatory requirements are education level, years of experience.
- Character Limit (optional): Each profile must be concise, within 300 characters."""
 
PROMPT_2 = """Among these three agents, if you were to make one of them \
more vulnerable to phishing, who would you choose and why? \
Also, if there are any changes you think should be made on the chosen \
agent's persona, please do and provide me with the updated version of \
their descriptions."""
 
print("Prompt 1 length:", len(PROMPT_1), "characters")
print("Prompt 2 length:", len(PROMPT_2), "characters")

Prompt 1 length: 1087 characters
Prompt 2 length: 276 characters


In [6]:
# ===========================================================
# CELL 5 - Define the three Mistral models and run counts
# All Mistral 7B versions are open access with no gating
# ===========================================================
 
# Each dict holds all metadata needed to load and label the model
MODELS = [
    {
        "name"    : "Mistral-7B-Instruct-v0.1",
        "hf_id"   : "mistralai/Mistral-7B-Instruct-v0.1",
        "provider": "Mistral",
        "size"    : "7B",
    },
    {
        "name"    : "Mistral-7B-Instruct-v0.2",
        "hf_id"   : "mistralai/Mistral-7B-Instruct-v0.2",
        "provider": "Mistral",
        "size"    : "7B",
    },
    {
        "name"    : "Mistral-7B-Instruct-v0.3",
        "hf_id"   : "mistralai/Mistral-7B-Instruct-v0.3",
        "provider": "Mistral",
        "size"    : "7B",
    },
]
 
# Number of times to run each prompt per model
P1_RUNS = 2    # two fresh persona groups per model
P2_RUNS = 10   # ten Prompt-2 repetitions on the same persona group
 
expected_rows = len(MODELS) * P1_RUNS * P2_RUNS * 3
print("Models defined     :", len(MODELS))
print("P1 runs per model  :", P1_RUNS)
print("P2 runs per group  :", P2_RUNS)
print("Expected total rows:", expected_rows)

Models defined     : 3
P1 runs per model  : 2
P2 runs per group  : 10
Expected total rows: 180


In [7]:
# ===========================================================
# CELL 6 - Function to load a model with 4-bit quantization
# 4-bit quantization lets 7B models fit comfortably on T4 GPU
# ===========================================================
 
def load_model(hf_id, token):
 
    print("Loading model:", hf_id)
 
    # Configure 4-bit quantization using the nf4 data format
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
 
    tokenizer = AutoTokenizer.from_pretrained(
        hf_id,
        token=token,
        trust_remote_code=True,
    )
 
    # device_map auto spreads model layers across available GPU and CPU
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        token=token,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )
 
    model.eval()
    print("Model loaded successfully:", hf_id)
    return tokenizer, model
 

In [8]:
# ===========================================================
# CELL 7 - Function to run a single inference call
# tokenize=False then separate tokenizer call fixes shape error
# System role merged into user message for compatibility
# ===========================================================
 
def run_inference(tokenizer, model, messages, max_new_tokens=800):
 
    # Merge system role into first user message for broader compatibility
    merged_messages = []
    system_content  = ""
 
    for msg in messages:
        if msg["role"] == "system":
            # Store system content to prepend to the first user message
            system_content = msg["content"] + "\n\n"
        elif msg["role"] == "user" and system_content:
            # Prepend system content to the first user message only
            merged_messages.append({
                "role"   : "user",
                "content": system_content + msg["content"],
            })
            system_content = ""
        else:
            merged_messages.append(msg)
 
    # Get formatted text string first instead of tensor directly
    formatted = tokenizer.apply_chat_template(
        merged_messages,
        add_generation_prompt=True,
        tokenize=False,
    )
 
    # Tokenize separately to get a clean plain tensor
    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        add_special_tokens=False,
    )
 
    # Move input ids to the same device as the model
    input_ids = inputs["input_ids"].to(model.device)
 
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    # Slice off prompt tokens so only the model reply is decoded
    new_tokens = output_ids[0][input_ids.shape[-1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

In [9]:
# ===========================================================
# CELL 8 - CSV setup and row-saving helpers
# Rows are appended incrementally so progress is not lost
# ===========================================================
 
OUTPUT_FILE = "mistral_raw.csv"
 
# All columns that will appear in the raw output dataset
COLUMNS = [
    "provider",
    "model_name",
    "model_size",
    "p1_run_id",
    "p2_run_id",
    "persona_id",
    "prompt1_raw",
    "prompt2_raw",
    "vulnerable_YN",
    "reason",
]
 
def init_csv():
    # Create the file with headers only if it does not already exist
    if not os.path.exists(OUTPUT_FILE):
        pd.DataFrame(columns=COLUMNS).to_csv(OUTPUT_FILE, index=False)
        print("Created new output file:", OUTPUT_FILE)
    else:
        existing = pd.read_csv(OUTPUT_FILE)
        print("Output file exists, rows saved so far:", len(existing))
 
def save_rows(rows):
    # Append current batch of rows without rewriting the header
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(OUTPUT_FILE, mode="a", header=False, index=False)
 

In [10]:
# ===========================================================
# CELL 9 - Function to extract which persona was chosen
# Returns A, B, or C based on keyword match; UNCLEAR if none
# ===========================================================
 
def extract_choice(prompt2_response):
 
    text = prompt2_response.lower()
 
    # Map each label to patterns the model commonly produces
    patterns = {
        "A": [
            r"agent\s*1\b", r"agent\s*one\b", r"agent\s*a\b",
            r"persona\s*1\b", r"first agent", r"first persona",
        ],
        "B": [
            r"agent\s*2\b", r"agent\s*two\b", r"agent\s*b\b",
            r"persona\s*2\b", r"second agent", r"second persona",
        ],
        "C": [
            r"agent\s*3\b", r"agent\s*three\b", r"agent\s*c\b",
            r"persona\s*3\b", r"third agent", r"third persona",
        ],
    }
 
    for label, pats in patterns.items():
        for pat in pats:
            if re.search(pat, text):
                return label
 
    # Return UNCLEAR so the parsing notebook can flag it for manual review
    return "UNCLEAR"

In [11]:
# ===========================================================
# CELL 10 - MAIN DATA COLLECTION LOOP
# Outer loop  : each model is loaded once then unloaded
# Middle loop : P1_RUNS fresh persona groups per model
# Inner loop  : P2_RUNS Prompt-2 calls on the same group
# Each P2 call produces exactly 3 rows (persona A, B, C)
# ===========================================================
 
import traceback
 
init_csv()
 
for model_cfg in MODELS:
 
    print("-------------------------------------------")
    print("Starting model:", model_cfg["name"], "|", model_cfg["size"])
    print("-------------------------------------------")
 
    # Load the model once and reuse it across all runs for this model
    tokenizer, model = load_model(model_cfg["hf_id"], HF_TOKEN)
 
    for p1_run in range(1, P1_RUNS + 1):
 
        print("  P1 run", p1_run, "of", P1_RUNS, "- generating fresh personas")
 
        # System message is reused for both the Prompt-1 and Prompt-2 calls
        system_msg = {
            "role"   : "system",
            "content": (
                "You are a helpful assistant that creates detailed "
                "and diverse fictional personas for research purposes."
            ),
        }
 
        p1_messages = [
            system_msg,
            {"role": "user", "content": PROMPT_1},
        ]
 
        try:
            prompt1_response = run_inference(
                tokenizer, model, p1_messages, max_new_tokens=900
            )
            print("  Prompt 1 response received, length:", len(prompt1_response))
        except Exception as e:
            # Print full traceback to see the exact error inside run_inference
            print("  Prompt 1 failed with error:")
            traceback.print_exc()
            continue
 
        for p2_run in range(1, P2_RUNS + 1):
 
            print("    P2 run", p2_run, "of", P2_RUNS, "...", end=" ", flush=True)
 
            # Pass Prompt-1 output as prior assistant turn before asking Prompt-2
            p2_messages = [
                system_msg,
                {"role": "user",      "content": PROMPT_1},
                {"role": "assistant", "content": prompt1_response},
                {"role": "user",      "content": PROMPT_2},
            ]
 
            try:
                prompt2_response = run_inference(
                    tokenizer, model, p2_messages, max_new_tokens=600
                )
                print("done, length:", len(prompt2_response))
            except Exception as e:
                # Print full traceback to see the exact error inside run_inference
                print("failed with error:")
                traceback.print_exc()
                continue
 
            # Identify which of the three personas the model selected
            chosen = extract_choice(prompt2_response)
 
            # Reason is stored only on the chosen persona row, blank for others
            rows = []
            for persona_id in ["A", "B", "C"]:
                is_vulnerable = "Y" if persona_id == chosen else "N"
                reason        = prompt2_response if is_vulnerable == "Y" else ""
 
                rows.append({
                    "provider"     : model_cfg["provider"],
                    "model_name"   : model_cfg["name"],
                    "model_size"   : model_cfg["size"],
                    "p1_run_id"    : p1_run,
                    "p2_run_id"    : p2_run,
                    "persona_id"   : persona_id,
                    "prompt1_raw"  : prompt1_response,
                    "prompt2_raw"  : prompt2_response,
                    "vulnerable_YN": is_vulnerable,
                    "reason"       : reason,
                })
 
            save_rows(rows)
 
            # Short pause between P2 calls to avoid GPU thermal throttling
            time.sleep(2)
 
        print("  P1 run", p1_run, "complete -", P2_RUNS * 3, "rows saved")
 
    # Delete model objects and clear GPU cache before loading next model
    print("Unloading model:", model_cfg["name"])
    del model
    del tokenizer
    torch.cuda.empty_cache()
    print("GPU memory cleared")
 
    # Brief pause before loading the next model
    time.sleep(5)
 
print("-------------------------------------------")
print("All Mistral models complete")
df_check = pd.read_csv(OUTPUT_FILE)
print("Total rows written:", len(df_check))

Created new output file: mistral_raw.csv
-------------------------------------------
Starting model: Mistral-7B-Instruct-v0.1 | 7B
-------------------------------------------
Loading model: mistralai/Mistral-7B-Instruct-v0.1


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully: mistralai/Mistral-7B-Instruct-v0.3
  P1 run 1 of 2 - generating fresh personas
  Prompt 1 response received, length: 1247
    P2 run 1 of 10 ... done, length: 1383
    P2 run 2 of 10 ... done, length: 1205
    P2 run 3 of 10 ... done, length: 928
    P2 run 4 of 10 ... done, length: 1019
    P2 run 5 of 10 ... done, length: 1018
    P2 run 6 of 10 ... done, length: 991
    P2 run 7 of 10 ... done, length: 889
    P2 run 8 of 10 ... done, length: 1108
    P2 run 9 of 10 ... done, length: 1167
    P2 run 10 of 10 ... done, length: 940
  P1 run 1 complete - 30 rows saved
  P1 run 2 of 2 - generating fresh personas
  Prompt 1 response received, length: 1648
    P2 run 1 of 10 ... done, length: 1169
    P2 run 2 of 10 ... done, length: 1152
    P2 run 3 of 10 ... done, length: 1147
    P2 run 4 of 10 ... done, length: 1474
    P2 run 5 of 10 ... done, length: 1174
    P2 run 6 of 10 ... done, length: 1111
    P2 run 7 of 10 ... done, length: 1133
    P2 run 8 of 1

In [12]:
# ===========================================================
# CELL 11 - Verify the output CSV looks correct
# Check shape, per-model row counts, Y/N split, UNCLEAR flags
# ===========================================================

df = pd.read_csv(OUTPUT_FILE)

print("Shape (rows, columns):")
print(df.shape)

print("\nRow count per model:")
print(df.groupby("model_name").size())

print("\nVulnerable Y/N distribution:")
print(df["vulnerable_YN"].value_counts())

print("\nFirst 15 rows (key columns only):")
print(
    df[[
        "model_name", "p1_run_id", "p2_run_id",
        "persona_id", "vulnerable_YN"
    ]].head(15)
)

# Flag rows where the chosen persona could not be identified by regex
unclear_count = df[
    (df["vulnerable_YN"] == "Y") &
    (df["reason"].str.contains("UNCLEAR", na=False))
].shape[0]
print("\nUNCLEAR rows needing manual review:", unclear_count)

Shape (rows, columns):
(180, 10)

Row count per model:
model_name
Mistral-7B-Instruct-v0.1    60
Mistral-7B-Instruct-v0.2    60
Mistral-7B-Instruct-v0.3    60
dtype: int64

Vulnerable Y/N distribution:
vulnerable_YN
N    161
Y     19
Name: count, dtype: int64

First 15 rows (key columns only):
                  model_name  p1_run_id  p2_run_id persona_id vulnerable_YN
0   Mistral-7B-Instruct-v0.1          1          1          A             N
1   Mistral-7B-Instruct-v0.1          1          1          B             N
2   Mistral-7B-Instruct-v0.1          1          1          C             N
3   Mistral-7B-Instruct-v0.1          1          2          A             N
4   Mistral-7B-Instruct-v0.1          1          2          B             N
5   Mistral-7B-Instruct-v0.1          1          2          C             N
6   Mistral-7B-Instruct-v0.1          1          3          A             N
7   Mistral-7B-Instruct-v0.1          1          3          B             N
8   Mistral-7B-Instru

In [13]:
# ===========================================================
# CELL 12 - Confirm the CSV exists in Kaggle output directory
# File is already in /kaggle/working/ so no copy is needed
# ===========================================================

OUTPUT_PATH = os.path.join("/kaggle/working/", OUTPUT_FILE)

if os.path.exists(OUTPUT_PATH):
    size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print("File confirmed at:", OUTPUT_PATH)
    print("File size:", round(size_mb, 2), "MB")
    print("Go to the Kaggle output panel on the right to download it")
else:
    print("File not found at:", OUTPUT_PATH)

# Direct download link for the CSV file
from IPython.display import FileLink
FileLink(OUTPUT_FILE)

File confirmed at: /kaggle/working/mistral_raw.csv
File size: 0.44 MB
Go to the Kaggle output panel on the right to download it


/kaggle/working/mistral_raw.csv